# 04b — Bán giám sát (Semi-supervised Learning)

Mục tiêu:
- Giả lập kịch bản thiếu nhãn (5%, 10%, 20%, 30%)
- So sánh Supervised-only vs Self-Training tại mỗi tỷ lệ
- Vẽ Learning Curve
- Phân tích rủi ro Pseudo-label (tác động chính sách HR)

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from src.data.loader import load_data, load_config
from src.data.cleaner import HRDataCleaner
from src.features.builder import FeatureBuilder
from src.models.semi_supervised import (
    run_label_ratio_experiment, analyze_pseudo_label_risk,
    mask_labels, train_semi_supervised,
)
from src.evaluation.metrics import evaluate_classifier
from src.visualization.plots import plot_learning_curve_semi

config = load_config("configs/params.yaml")

## 4b.1 Chuẩn bị dữ liệu

In [ ]:
df = load_data(config["data"]["raw_path"])
cleaner = HRDataCleaner(target_col="Attrition")
df_clean = cleaner.clean(df)
fb = FeatureBuilder(df_clean)
df_featured = fb.build_all()
df_encoded = cleaner.encode(df_featured)

X = df_encoded.drop(columns=["Attrition"])
y = df_encoded["Attrition"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)

## 4b.2 Thí nghiệm Multi-ratio: Supervised vs Self-Training

Giả lập kịch bản thiếu nhãn:
- 5% nhãn (ẩn 95%)
- 10% nhãn (ẩn 90%)
- 20% nhãn (ẩn 80%)
- 30% nhãn (ẩn 70%)

In [ ]:
ratios = config["semi_supervised"]["label_ratios"]

semi_curve = run_label_ratio_experiment(
    X_train.values, y_train.values,
    X_test.values, y_test.values,
    ratios=ratios,
    n_estimators=config["model"]["rf_n_estimators"],
    random_state=config["model"]["random_state"],
)

## 4b.3 Learning Curve: % nhãn vs PR-AUC/F1

> Learning curve cho thấy Self-Training cải thiện hiệu suất khi ít nhãn, nhưng khoảng cách thu hẹp khi nhiều nhãn hơn.

In [ ]:
plot_learning_curve_semi(semi_curve)

## 4b.4 Bảng kết quả chi tiết

In [ ]:
semi_curve["results_df"]

## 4b.5 Phân tích rủi ro Pseudo-label

> **Câu hỏi quan trọng**: Pseudo-labels có đáng tin cậy không? Nếu gán sai nhãn 'sẽ nghỉ' cho người ở lại → lãng phí ngân sách can thiệp. Nếu bỏ sót người 'sẽ nghỉ' → mất nhân tài.

In [ ]:
pseudo_risk = analyze_pseudo_label_risk(
    X_train.values, y_train.values,
    ratio=0.20,
    n_estimators=config["model"]["rf_n_estimators"],
    random_state=config["model"]["random_state"],
)

print(f"\nTổng pseudo-labels: {pseudo_risk['total_pseudo']}")
print(f"Accuracy: {pseudo_risk['accuracy']:.1%}")
print(f"False Positive Rate: {pseudo_risk['false_positive_rate']:.1%}")
print(f"False Negative Rate: {pseudo_risk['false_negative_rate']:.1%}")

## 4b.6 Nhận xét & Tác động chính sách

**Kết quả:**
- Self-Training cải thiện đáng kể khi chỉ có 5-10% nhãn
- Khi có 20-30% nhãn, sự cải thiện nhỏ hơn
- Pseudo-label accuracy cho thấy mức độ tin cậy

**Tác động chính sách HR:**
- Khi không có đủ dữ liệu khảo sát (nhãn thực), Self-Training giúp mở rộng dự đoán
- **Rủi ro False Positive**: gán nhãn 'sẽ nghỉ' cho NV ở lại → tốn chi phí can thiệp không cần thiết
- **Rủi ro False Negative**: bỏ sót NV thực sự sẽ nghỉ → mất nhân tài
- → Nên kết hợp Self-Training với kiểm tra thủ công cho các trường hợp có xác suất gần 0.5